In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.utils import resample   

FORM_FOLDERS = {
    "Fixed":       "results_ablation_fixed",
    "Linear":      "results_ablation_linear",
    "Square root": "results_ablation_sqrt",
    "Log":         "results_ablation_log",
}
ROUTERS          = ["bge-large-en-v1.5", "Reason-ModernColBERT", "Qwen3-Reranker-4B"]
ALL_K            = [3, 5, 8, 10, 12, 14]   
HELDOUT_K        = [10, 12, 14]            
NUMBER_OF_ROUNDS = 5000
SEED             = 42      

print("Forms:           ", list(FORM_FOLDERS))
print("Routers:         ", ROUTERS)
print("All K:           ", ALL_K)
print("Held-out K:      ", HELDOUT_K)
print("Bootstrap rounds:", NUMBER_OF_ROUNDS)

Forms:            ['Fixed', 'Linear', 'Square root', 'Log']
Routers:          ['bge-large-en-v1.5', 'Reason-ModernColBERT', 'Qwen3-Reranker-4B']
All K:            [3, 5, 8, 10, 12, 14]
Held-out K:       [10, 12, 14]
Bootstrap rounds: 5000


In [2]:
def load_one_file(form, router, k_value):
    folder = Path(FORM_FOLDERS[form]) / f"mmlu_test_eval_service_{k_value}_300"
    for file_path in folder.glob("*.csv"):
        if router.lower() in file_path.name.lower():
            return pd.read_csv(file_path)
    print("File not found:", form, router, k_value)


form_tables = {}
for form in FORM_FOLDERS:
    for router in ROUTERS:
        for k_value in ALL_K:
            form_tables[(form, router, k_value)] = load_one_file(form, router, k_value)

missing_files = [key for key, table in form_tables.items() if table is None]
print("Files loaded:", len(form_tables) - len(missing_files), "of 72")
print("Missing files:", len(missing_files))

Files loaded: 72 of 72
Missing files: 0


In [3]:
form_check_rows = []
for router in ROUTERS:
    for k_value in ALL_K:
        log_table = form_tables[("Log", router, k_value)]
        form_check_row = {"Router": router, "K": k_value,
                          "Same order": all(log_table["query"].equals(form_tables[(form, router, k_value)]["query"])
                                            for form in FORM_FOLDERS)}
        for form in FORM_FOLDERS:
            form_check_row[form] = form_tables[(form, router, k_value)]["is_correct"].mean() * 100
        form_check_rows.append(form_check_row)

form_check_table = pd.DataFrame(form_check_rows).round(2)
display(form_check_table)

,Router,K,Same order,Fixed,Linear,Square root,Log
0,bge-large-en-v1.5,3,True,91.83,92.50,92.00,91.67
1,bge-large-en-v1.5,5,True,88.50,88.88,88.88,89.00
2,bge-large-en-v1.5,8,True,80.00,82.36,82.36,82.82
3,bge-large-en-v1.5,10,True,78.15,80.38,80.62,80.77
4,bge-large-en-v1.5,12,True,68.93,70.93,71.67,71.27
5,bge-large-en-v1.5,14,True,66.06,67.53,68.65,69.00
6,Reason-ModernColBERT,3,True,91.83,92.00,92.00,91.67
7,Reason-ModernColBERT,5,True,89.38,89.25,89.62,89.88
8,Reason-ModernColBERT,8,True,82.45,83.64,83.55,83.55
9,Reason-ModernColBERT,10,True,78.92,80.38,80.62,80.46


In [4]:
def balanced_accuracy(correct, is_general):
    specialist_recall = correct[~is_general].mean()
    general_recall    = correct[is_general].mean()
    return (specialist_recall + general_recall) / 2


form_results = []

for router in ROUTERS:
    for k_value in HELDOUT_K:
        fixed_table   = form_tables[("Fixed", router, k_value)]
        correct_fixed = fixed_table["is_correct"].astype(int).to_numpy()
        is_general    = (fixed_table["expected_service"] == "general").to_numpy()
        all_rows      = np.arange(len(correct_fixed))

        for form in ["Linear", "Square root", "Log"]:
            correct_form = form_tables[(form, router, k_value)]["is_correct"].astype(int).to_numpy()

            observed = (balanced_accuracy(correct_form, is_general)
                        - balanced_accuracy(correct_fixed, is_general)) * 100

            print(f"{router}, K = {k_value}, {form} vs Fixed")

            differences = []
            for round_number in range(NUMBER_OF_ROUNDS):
                picked = resample(all_rows, replace=True, stratify=is_general, random_state=SEED + round_number)
                differences.append((balanced_accuracy(correct_form[picked], is_general[picked])
                                    - balanced_accuracy(correct_fixed[picked], is_general[picked])) * 100)

                if round_number == 0:
                    print(f"  round 1 picked {len(picked)} rows = "
                          f"{is_general[picked].sum()} general + {(~is_general[picked]).sum()} specialist "
                          f"({len(np.unique(picked))} different, {len(picked) - len(np.unique(picked))} repeats)")
                    print(f"  round 1 difference = {differences[-1]:.2f}")

            low, high = np.percentile(differences, [2.5, 97.5])
            print(f"  {NUMBER_OF_ROUNDS} rounds -> observed {observed:.2f}, 95% CI [{low:.2f}, {high:.2f}]\n")

            form_results.append({"Router": router, "Comparison": f"{form} - Fixed", "K": k_value,
                                 "Difference": observed, "CI low": low, "CI high": high})

form_results_table = pd.DataFrame(form_results).round(2)
display(form_results_table)

bge-large-en-v1.5, K = 10, Linear vs Fixed
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = 13.80
  5000 rounds -> observed 11.95, 95% CI [9.28, 14.63]

bge-large-en-v1.5, K = 10, Square root vs Fixed
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = 12.92
  5000 rounds -> observed 11.63, 95% CI [9.05, 14.32]

bge-large-en-v1.5, K = 10, Log vs Fixed
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = 12.83
  5000 rounds -> observed 11.38, 95% CI [8.82, 14.00]

bge-large-en-v1.5, K = 12, Linear vs Fixed
  round 1 picked 1500 rows = 300 general + 1200 specialist (946 different, 554 repeats)
  round 1 difference = 16.92
  5000 rounds -> observed 16.25, 95% CI [13.29, 19.25]

bge-large-en-v1.5, K = 12, Square root vs Fixed
  round 1 picked 1500 rows = 300 general + 1200 specialist (946 different, 554 repeats)
 

,Router,Comparison,K,Difference,CI low,CI high
0,bge-large-en-v1.5,Linear - Fixed,10,11.95,9.28,14.63
1,bge-large-en-v1.5,Square root - Fixed,10,11.63,9.05,14.32
2,bge-large-en-v1.5,Log - Fixed,10,11.38,8.82,14.00
3,bge-large-en-v1.5,Linear - Fixed,12,16.25,13.29,19.25
4,bge-large-en-v1.5,Square root - Fixed,12,15.71,12.79,18.62
5,bge-large-en-v1.5,Log - Fixed,12,14.08,11.29,16.96
6,bge-large-en-v1.5,Linear - Fixed,14,20.01,17.01,22.96
7,bge-large-en-v1.5,Square root - Fixed,14,18.60,15.60,21.46
8,bge-large-en-v1.5,Log - Fixed,14,17.50,14.67,20.35
9,Reason-ModernColBERT,Linear - Fixed,10,8.30,5.95,10.67


In [5]:
def balanced_accuracy(correct, is_general):
    specialist_recall = correct[~is_general].mean()
    general_recall    = correct[is_general].mean()
    return (specialist_recall + general_recall) / 2


form_results = []

for router in ROUTERS:
    for k_value in HELDOUT_K:
        correct_fixed = form_tables[("Fixed", router, k_value)]["is_correct"].astype(int).to_numpy()
        is_general    = (form_tables[("Fixed", router, k_value)]["expected_service"] == "general").to_numpy()
        all_rows      = np.arange(len(correct_fixed))

        for other_form in ["Linear", "Square root", "Log"]:
            correct_other = form_tables[(other_form, router, k_value)]["is_correct"].astype(int).to_numpy()

            observed = (balanced_accuracy(correct_other, is_general)
                        - balanced_accuracy(correct_fixed, is_general)) * 100

            print(f"{router}, K = {k_value}, {other_form} vs Fixed")

            differences = []
            for round_number in range(NUMBER_OF_ROUNDS):
                picked = resample(all_rows, replace=True, stratify=is_general, random_state=SEED + round_number)
                differences.append((balanced_accuracy(correct_other[picked], is_general[picked])
                                    - balanced_accuracy(correct_fixed[picked], is_general[picked])) * 100)

                # show how round 1 was built
                if round_number == 0:
                    print(f"  round 1 picked {len(picked)} rows = "
                          f"{is_general[picked].sum()} general + {(~is_general[picked]).sum()} specialist "
                          f"({len(np.unique(picked))} different, {len(picked) - len(np.unique(picked))} repeats)")
                    print(f"  round 1 difference = {differences[-1]:.2f}")

            low, high = np.percentile(differences, [2.5, 97.5])

            print(f"  {NUMBER_OF_ROUNDS} rounds -> observed {observed:.2f}, 95% CI [{low:.2f}, {high:.2f}]\n")

            form_results.append({"Router": router, "Comparison": f"{other_form} - Fixed", "K": k_value,
                                 "Difference": observed, "CI low": low, "CI high": high})

form_results_table = pd.DataFrame(form_results).round(2)
display(form_results_table)

bge-large-en-v1.5, K = 10, Linear vs Fixed
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = 13.80
  5000 rounds -> observed 11.95, 95% CI [9.28, 14.63]

bge-large-en-v1.5, K = 10, Square root vs Fixed
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = 12.92
  5000 rounds -> observed 11.63, 95% CI [9.05, 14.32]

bge-large-en-v1.5, K = 10, Log vs Fixed
  round 1 picked 1300 rows = 300 general + 1000 specialist (799 different, 501 repeats)
  round 1 difference = 12.83
  5000 rounds -> observed 11.38, 95% CI [8.82, 14.00]

bge-large-en-v1.5, K = 12, Linear vs Fixed
  round 1 picked 1500 rows = 300 general + 1200 specialist (946 different, 554 repeats)
  round 1 difference = 16.92
  5000 rounds -> observed 16.25, 95% CI [13.29, 19.25]

bge-large-en-v1.5, K = 12, Square root vs Fixed
  round 1 picked 1500 rows = 300 general + 1200 specialist (946 different, 554 repeats)
 

,Router,Comparison,K,Difference,CI low,CI high
0,bge-large-en-v1.5,Linear - Fixed,10,11.95,9.28,14.63
1,bge-large-en-v1.5,Square root - Fixed,10,11.63,9.05,14.32
2,bge-large-en-v1.5,Log - Fixed,10,11.38,8.82,14.00
3,bge-large-en-v1.5,Linear - Fixed,12,16.25,13.29,19.25
4,bge-large-en-v1.5,Square root - Fixed,12,15.71,12.79,18.62
5,bge-large-en-v1.5,Log - Fixed,12,14.08,11.29,16.96
6,bge-large-en-v1.5,Linear - Fixed,14,20.01,17.01,22.96
7,bge-large-en-v1.5,Square root - Fixed,14,18.60,15.60,21.46
8,bge-large-en-v1.5,Log - Fixed,14,17.50,14.67,20.35
9,Reason-ModernColBERT,Linear - Fixed,10,8.30,5.95,10.67
